# MLflow Pipeline Validation Test

**Purpose:** Validate that our data analysis pipeline produces meaningful, non-spurious results.

This notebook tests:
1. **Data Ingestion** - Can we load and parse the actual datathon data?
2. **Feature Engineering** - Are derived metrics computed correctly?
3. **Model Training** - Do models learn real patterns (not just noise)?
4. **Cross-Validation** - Are metrics stable across folds?
5. **Baseline Comparisons** - Do our models beat naive baselines?
6. **Sanity Checks** - Are feature importances interpretable?

**Success Criteria:**
- Models must beat a naive mean/median baseline
- Cross-validation variance should be reasonable (not wildly unstable)
- Feature importances should align with domain knowledge
- No data leakage detected

---

In [ ]:
# ============================================================================
# CELL 0: Environment Setup
# ============================================================================

import warnings
warnings.filterwarnings('ignore')

# Core dependencies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
import json
import tempfile
import os

# ML dependencies
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.dummy import DummyRegressor

# MLflow
try:
    import mlflow
    import mlflow.sklearn
    MLFLOW_AVAILABLE = True
    print(f"✅ MLflow version: {mlflow.__version__}")
except ImportError:
    print("⚠️ MLflow not installed. Installing...")
    !pip install -q mlflow
    import mlflow
    import mlflow.sklearn
    MLFLOW_AVAILABLE = True
    print(f"✅ MLflow installed: {mlflow.__version__}")

# Optional: XGBoost
try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
    print("✅ XGBoost available")
except ImportError:
    XGBOOST_AVAILABLE = False
    print("⚠️ XGBoost not available, using GradientBoosting")

# Plotting
try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except ImportError:
    sns = None

plt.rcParams.update({"figure.figsize": (10, 6)})

print("\n✅ All dependencies loaded")

In [ ]:
# ============================================================================
# CELL 1: MLflow Experiment Setup
# ============================================================================

EXPERIMENT_NAME = "DSC_Datathon_2026_Pipeline_Validation"

# Set tracking URI (local for testing, Databricks will override)
# mlflow.set_tracking_uri("mlruns")  # Uncomment for local file-based tracking

try:
    experiment_id = mlflow.create_experiment(
        EXPERIMENT_NAME,
        tags={
            "purpose": "pipeline_validation",
            "team": "dvislawa",
            "test_date": datetime.now().strftime("%Y-%m-%d"),
        }
    )
    print(f"✅ Created experiment: {EXPERIMENT_NAME}")
except mlflow.exceptions.MlflowException:
    experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    experiment_id = experiment.experiment_id
    print(f"✅ Using existing experiment: {EXPERIMENT_NAME}")

mlflow.set_experiment(EXPERIMENT_NAME)
print(f"   Experiment ID: {experiment_id}")

In [ ]:
# ============================================================================
# CELL 2: Data Ingestion Test
# ============================================================================

print("="*70)
print("TEST 1: DATA INGESTION")
print("="*70)

# Find data directory
DATA_CANDIDATES = [
    Path("../data/geo_mismatch"),
    Path("data/geo_mismatch"),
    Path("../../data/geo_mismatch"),
]
DATA_DIR = next((p for p in DATA_CANDIDATES if p.exists()), None)

if DATA_DIR is None:
    raise FileNotFoundError("Could not find data/geo_mismatch directory")

print(f"Data directory: {DATA_DIR.resolve()}")

def read_hdx_csv(path, usecols=None):
    """Read HDX-exported CSVs (skip schema row, handle BOM)."""
    return pd.read_csv(path, skiprows=[1], encoding="utf-8-sig", usecols=usecols, low_memory=False)

# Test loading each key dataset
ingestion_results = {}

# HNO Data
try:
    hno_2026 = read_hdx_csv(DATA_DIR / "hpc_hno_2026.csv")
    ingestion_results["HNO 2026"] = {"status": "✅", "rows": len(hno_2026), "cols": len(hno_2026.columns)}
except Exception as e:
    ingestion_results["HNO 2026"] = {"status": "❌", "error": str(e)}

# HRP Data
try:
    hrp = read_hdx_csv(DATA_DIR / "humanitarian-response-plans.csv")
    ingestion_results["HRP"] = {"status": "✅", "rows": len(hrp), "cols": len(hrp.columns)}
except Exception as e:
    ingestion_results["HRP"] = {"status": "❌", "error": str(e)}

# INFORM Data
try:
    inform = pd.read_csv(DATA_DIR / "inform_severity_master_2020_2025.csv", encoding="utf-8-sig")
    ingestion_results["INFORM"] = {"status": "✅", "rows": len(inform), "cols": len(inform.columns)}
except Exception as e:
    ingestion_results["INFORM"] = {"status": "❌", "error": str(e)}

# Population Data
try:
    pop = read_hdx_csv(DATA_DIR / "cod_population_admin0.csv")
    ingestion_results["COD Population"] = {"status": "✅", "rows": len(pop), "cols": len(pop.columns)}
except Exception as e:
    ingestion_results["COD Population"] = {"status": "❌", "error": str(e)}

# Print results
print("\nIngestion Results:")
for dataset, result in ingestion_results.items():
    if result["status"] == "✅":
        print(f"  {result['status']} {dataset}: {result['rows']:,} rows × {result['cols']} cols")
    else:
        print(f"  {result['status']} {dataset}: {result.get('error', 'Unknown error')}")

# Log to MLflow
with mlflow.start_run(run_name="Test_1_Data_Ingestion"):
    mlflow.set_tag("test_type", "ingestion")
    for dataset, result in ingestion_results.items():
        mlflow.log_param(f"{dataset.replace(' ', '_')}_status", result["status"])
        if "rows" in result:
            mlflow.log_metric(f"{dataset.replace(' ', '_')}_rows", result["rows"])
    
    all_passed = all(r["status"] == "✅" for r in ingestion_results.values())
    mlflow.log_metric("ingestion_pass_rate", 1.0 if all_passed else 0.0)
    print(f"\n{'✅ ALL INGESTION TESTS PASSED' if all_passed else '❌ SOME INGESTION TESTS FAILED'}")

In [ ]:
# ============================================================================
# CELL 3: Feature Engineering Validation
# ============================================================================

print("="*70)
print("TEST 2: FEATURE ENGINEERING")
print("="*70)

# Load all HNO years
YEARS = [2024, 2025, 2026]
HNO_COLS = ["Country ISO3", "Description", "Cluster", "Category", "Population", "In Need", "Targeted"]

hno = pd.concat([
    read_hdx_csv(DATA_DIR / f"hpc_hno_{y}.csv", usecols=HNO_COLS).assign(year=y)
    for y in YEARS
], ignore_index=True)

for c in ["Population", "In Need", "Targeted"]:
    hno[c] = pd.to_numeric(hno[c], errors="coerce")

# Extract overall caseload (Cluster='ALL', Category blank)
hno["Cluster"] = hno["Cluster"].astype(str).str.strip()
hno["Category"] = hno["Category"].fillna("").astype(str).str.strip()

hno_overall = (
    hno.query("Cluster == 'ALL' and Category == ''")
    .rename(columns={
        "Country ISO3": "iso3",
        "Population": "population",
        "In Need": "in_need",
        "Targeted": "targeted",
    })
    [["year", "iso3", "population", "in_need", "targeted"]]
    .copy()
)

print(f"HNO overall records: {len(hno_overall)}")

# Feature engineering tests
feature_tests = {}

# Test 1: need_rate calculation
hno_overall["need_rate"] = hno_overall["in_need"] / hno_overall["population"]
valid_need_rate = hno_overall["need_rate"].between(0, 1, inclusive="both").sum()
feature_tests["need_rate_valid"] = valid_need_rate / len(hno_overall.dropna(subset=["need_rate"]))

# Test 2: coverage_rate calculation
hno_overall["coverage_rate"] = hno_overall["targeted"] / hno_overall["in_need"]
valid_coverage = hno_overall["coverage_rate"].between(0, 2, inclusive="both").sum()  # Allow up to 200%
feature_tests["coverage_rate_valid"] = valid_coverage / len(hno_overall.dropna(subset=["coverage_rate"]))

# Test 3: No impossible values
impossible_values = (
    (hno_overall["in_need"] < 0).sum() +
    (hno_overall["population"] < 0).sum() +
    (hno_overall["targeted"] < 0).sum()
)
feature_tests["no_impossible_values"] = 1.0 if impossible_values == 0 else 0.0

# Test 4: Percentile rank stability
for year in YEARS:
    year_df = hno_overall[hno_overall["year"] == year].copy()
    year_df["need_rate_pct"] = year_df["need_rate"].rank(pct=True)
    # Percentiles should span 0-1
    pct_range = year_df["need_rate_pct"].max() - year_df["need_rate_pct"].min()
    feature_tests[f"pct_rank_range_{year}"] = pct_range

# Print results
print("\nFeature Engineering Tests:")
for test_name, value in feature_tests.items():
    status = "✅" if value >= 0.95 else "⚠️" if value >= 0.80 else "❌"
    print(f"  {status} {test_name}: {value:.3f}")

# Log to MLflow
with mlflow.start_run(run_name="Test_2_Feature_Engineering"):
    mlflow.set_tag("test_type", "feature_engineering")
    for test_name, value in feature_tests.items():
        mlflow.log_metric(test_name, value)
    
    all_passed = all(v >= 0.80 for v in feature_tests.values())
    mlflow.log_metric("feature_engineering_pass", 1.0 if all_passed else 0.0)
    print(f"\n{'✅ FEATURE ENGINEERING VALIDATED' if all_passed else '⚠️ SOME FEATURE TESTS NEED REVIEW'}")

In [ ]:
# ============================================================================
# CELL 4: Build Analysis Dataset (Geo-Insight)
# ============================================================================

print("="*70)
print("TEST 3: ANALYSIS DATASET CONSTRUCTION")
print("="*70)

def split_pipe_list(x):
    if pd.isna(x):
        return []
    return [p.strip() for p in str(x).split("|") if p.strip()]

# Load HRP
HRP_COLS = ["code", "startDate", "endDate", "locations", "years", "origRequirements", "revisedRequirements"]
hrp = read_hdx_csv(DATA_DIR / "humanitarian-response-plans.csv", usecols=HRP_COLS)

for c in ["origRequirements", "revisedRequirements"]:
    hrp[c] = pd.to_numeric(hrp[c], errors="coerce")

hrp["startDate"] = pd.to_datetime(hrp["startDate"], errors="coerce")
hrp["loc_list"] = hrp["locations"].apply(split_pipe_list)
hrp["year_list"] = hrp["years"].apply(split_pipe_list)
hrp["n_locations"] = hrp["loc_list"].map(len)

# Filter to single-country plans
hrp_single = hrp.query("n_locations == 1").copy()
hrp_single = hrp_single.explode("year_list")
hrp_single["year"] = pd.to_numeric(hrp_single["year_list"], errors="coerce")
hrp_single = hrp_single[hrp_single["year"].isin(YEARS)].copy()
hrp_single["year"] = hrp_single["year"].astype(int)
hrp_single["iso3"] = hrp_single["loc_list"].str[0]

hrp_agg = (
    hrp_single.assign(revisedRequirements=hrp_single["revisedRequirements"].fillna(0))
    .groupby(["year", "iso3"], as_index=False)
    .agg(req_sum=("revisedRequirements", "sum"))
)

# Merge HNO + HRP
core = hno_overall.merge(hrp_agg, on=["year", "iso3"], how="left")
core["req_sum"] = core["req_sum"].fillna(0)

# Compute key metrics
core["usd_per_in_need"] = core["req_sum"] / core["in_need"]
core.loc[~np.isfinite(core["usd_per_in_need"]), "usd_per_in_need"] = np.nan

core["log10_usd_per_in_need"] = np.log10(core["usd_per_in_need"].where(core["usd_per_in_need"] > 0))
core["log10_in_need"] = np.log10(core["in_need"].where(core["in_need"] > 0))

# Percentile ranks within year
core["need_rate_pct"] = core.groupby("year")["need_rate"].rank(pct=True)
core["usd_per_in_need_pct"] = core.groupby("year")["usd_per_in_need"].rank(pct=True)
core["mismatch"] = core["need_rate_pct"] - core["usd_per_in_need_pct"]

# Load INFORM for enrichment
inform_raw = pd.read_csv(DATA_DIR / "inform_severity_master_2020_2025.csv", encoding="utf-8-sig")
inform = inform_raw.iloc[2:].copy()

inform_cols = {
    "ISO3": "iso3",
    "INFORM Severity Index": "severity_index",
    "Regions": "region",
    "Year": "year",
    "TYPE OF CRISIS": "crisis_type",
    "DRIVERS": "drivers",
}
inform = inform[list(inform_cols.keys())].rename(columns=inform_cols)
inform["severity_index"] = pd.to_numeric(inform["severity_index"], errors="coerce")
inform["year"] = pd.to_numeric(inform["year"], errors="coerce")
inform = inform[~inform["iso3"].str.contains(",", na=False)].copy()

# Aggregate INFORM by iso3/year
inform_agg = inform.groupby(["iso3", "year"], as_index=False).agg({
    "severity_index": "max",
    "region": "first",
    "crisis_type": "first",
})

# Use 2025 for 2026
inform_2026 = inform_agg[inform_agg["year"] == 2025].copy()
inform_2026["year"] = 2026
inform_for_join = pd.concat([inform_agg[inform_agg["year"].isin([2024, 2025])], inform_2026], ignore_index=True)

# Merge
core_enriched = core.merge(inform_for_join, on=["iso3", "year"], how="left")

# Dataset quality metrics
dataset_metrics = {
    "total_rows": len(core_enriched),
    "unique_countries": core_enriched["iso3"].nunique(),
    "unique_years": core_enriched["year"].nunique(),
    "hrp_coverage": (core_enriched["req_sum"] > 0).mean(),
    "inform_coverage": core_enriched["severity_index"].notna().mean(),
    "target_available": core_enriched["log10_usd_per_in_need"].notna().mean(),
}

print("\nAnalysis Dataset Metrics:")
for metric, value in dataset_metrics.items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.1%}" if value <= 1 else f"  {metric}: {value:.2f}")
    else:
        print(f"  {metric}: {value}")

# Log to MLflow
with mlflow.start_run(run_name="Test_3_Dataset_Construction"):
    mlflow.set_tag("test_type", "dataset_construction")
    for metric, value in dataset_metrics.items():
        mlflow.log_metric(metric, value)
    
    # Pass if we have reasonable coverage
    passed = dataset_metrics["target_available"] >= 0.5 and dataset_metrics["unique_countries"] >= 40
    mlflow.log_metric("dataset_construction_pass", 1.0 if passed else 0.0)
    print(f"\n{'✅ DATASET CONSTRUCTION VALIDATED' if passed else '⚠️ DATASET NEEDS REVIEW'}")

In [ ]:
# ============================================================================
# CELL 5: Model Validation - Baseline Comparison
# ============================================================================

print("="*70)
print("TEST 4: MODEL VALIDATION (BASELINE COMPARISON)")
print("="*70)

# Prepare modeling data
model_df = core_enriched.dropna(subset=["log10_usd_per_in_need", "need_rate", "log10_in_need"]).copy()
model_df["crisis_type_primary"] = model_df["crisis_type"].astype(str).str.split("|").str[0].str.strip()

print(f"Modeling dataset: {len(model_df)} rows")

# Train/test split by year
train_df = model_df[model_df["year"].isin([2024, 2025])].copy()
test_df = model_df[model_df["year"] == 2026].copy()

print(f"Train: {len(train_df)} rows (2024-2025)")
print(f"Test:  {len(test_df)} rows (2026)")

# Features
num_features = ["need_rate", "log10_in_need"]
cat_features = ["region"]

# Add severity if available
if model_df["severity_index"].notna().mean() > 0.5:
    num_features.append("severity_index")

# Filter to available features
num_features = [c for c in num_features if c in model_df.columns]
cat_features = [c for c in cat_features if c in model_df.columns and model_df[c].notna().mean() > 0.5]

print(f"\nFeatures: {num_features + cat_features}")

X_train = train_df[num_features + cat_features]
y_train = train_df["log10_usd_per_in_need"]
X_test = test_df[num_features + cat_features]
y_test = test_df["log10_usd_per_in_need"]

# Preprocessing
pre = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_features),
    ] + ([
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_features),
    ] if cat_features else [])
)

# ---- Baseline Models ----
baseline_results = {}

# 1. Naive Mean Baseline
dummy_mean = DummyRegressor(strategy="mean")
dummy_mean.fit(X_train, y_train)
baseline_results["Naive_Mean"] = {
    "train_r2": r2_score(y_train, dummy_mean.predict(X_train)),
    "test_r2": r2_score(y_test, dummy_mean.predict(X_test)),
    "test_mae": mean_absolute_error(y_test, dummy_mean.predict(X_test)),
}

# 2. Naive Median Baseline
dummy_median = DummyRegressor(strategy="median")
dummy_median.fit(X_train, y_train)
baseline_results["Naive_Median"] = {
    "train_r2": r2_score(y_train, dummy_median.predict(X_train)),
    "test_r2": r2_score(y_test, dummy_median.predict(X_test)),
    "test_mae": mean_absolute_error(y_test, dummy_median.predict(X_test)),
}

# ---- Our Models ----

# 3. Ridge Regression
ridge_pipe = Pipeline([("pre", pre), ("model", Ridge(alpha=1.0))])
ridge_pipe.fit(X_train, y_train)
baseline_results["Ridge"] = {
    "train_r2": r2_score(y_train, ridge_pipe.predict(X_train)),
    "test_r2": r2_score(y_test, ridge_pipe.predict(X_test)),
    "test_mae": mean_absolute_error(y_test, ridge_pipe.predict(X_test)),
}

# 4. Gradient Boosting
gb_pipe = Pipeline([("pre", pre), ("model", GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=42))])
gb_pipe.fit(X_train, y_train)
baseline_results["GradientBoosting"] = {
    "train_r2": r2_score(y_train, gb_pipe.predict(X_train)),
    "test_r2": r2_score(y_test, gb_pipe.predict(X_test)),
    "test_mae": mean_absolute_error(y_test, gb_pipe.predict(X_test)),
}

# Print comparison
print("\n" + "-"*60)
print(f"{'Model':<20} {'Train R²':>12} {'Test R²':>12} {'Test MAE':>12}")
print("-"*60)
for model_name, metrics in baseline_results.items():
    print(f"{model_name:<20} {metrics['train_r2']:>12.4f} {metrics['test_r2']:>12.4f} {metrics['test_mae']:>12.4f}")
print("-"*60)

# Calculate improvement over baseline
best_baseline_r2 = max(baseline_results["Naive_Mean"]["test_r2"], baseline_results["Naive_Median"]["test_r2"])
ridge_improvement = baseline_results["Ridge"]["test_r2"] - best_baseline_r2
gb_improvement = baseline_results["GradientBoosting"]["test_r2"] - best_baseline_r2

print(f"\nImprovement over naive baseline:")
print(f"  Ridge:            {ridge_improvement:+.4f} R²")
print(f"  GradientBoosting: {gb_improvement:+.4f} R²")

# Log to MLflow
with mlflow.start_run(run_name="Test_4_Baseline_Comparison"):
    mlflow.set_tag("test_type", "baseline_comparison")
    
    for model_name, metrics in baseline_results.items():
        for metric_name, value in metrics.items():
            mlflow.log_metric(f"{model_name}_{metric_name}", value)
    
    mlflow.log_metric("ridge_improvement_over_baseline", ridge_improvement)
    mlflow.log_metric("gb_improvement_over_baseline", gb_improvement)
    
    # Pass if models beat baseline
    models_beat_baseline = ridge_improvement > 0 or gb_improvement > 0
    mlflow.log_metric("models_beat_baseline", 1.0 if models_beat_baseline else 0.0)
    
    print(f"\n{'✅ MODELS BEAT NAIVE BASELINE' if models_beat_baseline else '❌ MODELS DO NOT BEAT BASELINE - POTENTIAL ISSUE'}")

In [ ]:
# ============================================================================
# CELL 6: Cross-Validation Stability Test
# ============================================================================

print("="*70)
print("TEST 5: CROSS-VALIDATION STABILITY")
print("="*70)

# Use all data for cross-validation
X_all = model_df[num_features + cat_features]
y_all = model_df["log10_usd_per_in_need"]

# Cross-validation
cv = KFold(n_splits=5, shuffle=True, random_state=42)

cv_results = {}

# Ridge CV
ridge_cv = Pipeline([("pre", pre), ("model", Ridge(alpha=1.0))])
ridge_scores = cross_val_score(ridge_cv, X_all, y_all, cv=cv, scoring="r2")
cv_results["Ridge"] = {
    "mean_r2": ridge_scores.mean(),
    "std_r2": ridge_scores.std(),
    "min_r2": ridge_scores.min(),
    "max_r2": ridge_scores.max(),
    "scores": ridge_scores,
}

# GradientBoosting CV
gb_cv = Pipeline([("pre", pre), ("model", GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=42))])
gb_scores = cross_val_score(gb_cv, X_all, y_all, cv=cv, scoring="r2")
cv_results["GradientBoosting"] = {
    "mean_r2": gb_scores.mean(),
    "std_r2": gb_scores.std(),
    "min_r2": gb_scores.min(),
    "max_r2": gb_scores.max(),
    "scores": gb_scores,
}

# Print results
print("\nCross-Validation Results (5-fold):")
print("-"*60)
print(f"{'Model':<20} {'Mean R²':>10} {'Std R²':>10} {'Range':>20}")
print("-"*60)
for model_name, metrics in cv_results.items():
    range_str = f"[{metrics['min_r2']:.3f}, {metrics['max_r2']:.3f}]"
    print(f"{model_name:<20} {metrics['mean_r2']:>10.4f} {metrics['std_r2']:>10.4f} {range_str:>20}")
print("-"*60)

# Stability assessment
max_std = max(cv_results["Ridge"]["std_r2"], cv_results["GradientBoosting"]["std_r2"])
stability_status = "✅ STABLE" if max_std < 0.15 else "⚠️ MODERATE" if max_std < 0.25 else "❌ UNSTABLE"

print(f"\nStability Assessment: {stability_status} (max std: {max_std:.4f})")

# Visualization
fig, ax = plt.subplots(figsize=(10, 5))

positions = [1, 2]
data = [cv_results["Ridge"]["scores"], cv_results["GradientBoosting"]["scores"]]
labels = ["Ridge", "GradientBoosting"]

bp = ax.boxplot(data, positions=positions, widths=0.6, patch_artist=True)
colors = ["#3b82f6", "#10b981"]
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_xticks(positions)
ax.set_xticklabels(labels)
ax.set_ylabel("R² Score")
ax.set_title("Cross-Validation R² Distribution (5-fold)\nNarrower boxes = more stable")
ax.axhline(0, color="red", linestyle="--", alpha=0.5, label="Baseline (R²=0)")
ax.legend()
plt.tight_layout()

# Log to MLflow
with mlflow.start_run(run_name="Test_5_CrossValidation_Stability"):
    mlflow.set_tag("test_type", "cv_stability")
    
    for model_name, metrics in cv_results.items():
        mlflow.log_metric(f"{model_name}_cv_mean_r2", metrics["mean_r2"])
        mlflow.log_metric(f"{model_name}_cv_std_r2", metrics["std_r2"])
        mlflow.log_metric(f"{model_name}_cv_min_r2", metrics["min_r2"])
        mlflow.log_metric(f"{model_name}_cv_max_r2", metrics["max_r2"])
    
    mlflow.log_metric("max_cv_std", max_std)
    mlflow.log_metric("cv_stability_pass", 1.0 if max_std < 0.25 else 0.0)
    
    # Log plot
    with tempfile.TemporaryDirectory() as tmpdir:
        fig.savefig(f"{tmpdir}/cv_boxplot.png", dpi=150, bbox_inches="tight")
        mlflow.log_artifact(f"{tmpdir}/cv_boxplot.png", "plots")

plt.show()

print(f"\n{'✅ CV STABILITY VALIDATED' if max_std < 0.25 else '⚠️ CV RESULTS SHOW HIGH VARIANCE'}")

In [ ]:
# ============================================================================
# CELL 7: Feature Importance Sanity Check
# ============================================================================

print("="*70)
print("TEST 6: FEATURE IMPORTANCE SANITY CHECK")
print("="*70)

# Fit Ridge on all data
ridge_final = Pipeline([("pre", pre), ("model", Ridge(alpha=1.0))])
ridge_final.fit(X_all, y_all)

# Extract feature names
feature_names = list(num_features)
if cat_features:
    try:
        ohe = ridge_final.named_steps["pre"].named_transformers_["cat"].named_steps["ohe"]
        feature_names.extend(ohe.get_feature_names_out(cat_features).tolist())
    except Exception:
        pass

# Get coefficients
coefs = ridge_final.named_steps["model"].coef_

# Align if needed
if len(coefs) != len(feature_names):
    print(f"⚠️ Feature count mismatch: {len(coefs)} coefficients vs {len(feature_names)} names")
    feature_names = [f"feature_{i}" for i in range(len(coefs))]

coef_df = pd.DataFrame({"feature": feature_names, "coef": coefs})
coef_df["abs_coef"] = coef_df["coef"].abs()
coef_df = coef_df.sort_values("abs_coef", ascending=False)

print("\nTop Feature Coefficients (Ridge):")
print("-"*50)
for _, row in coef_df.head(10).iterrows():
    direction = "↑" if row["coef"] > 0 else "↓"
    print(f"  {direction} {row['feature']:<30} {row['coef']:>10.4f}")
print("-"*50)

# Sanity checks on feature importance
sanity_checks = {}

# Check 1: Are features actually being used? (not all near-zero)
max_abs_coef = coef_df["abs_coef"].max()
sanity_checks["features_have_impact"] = max_abs_coef > 0.01

# Check 2: Is log10_in_need a strong predictor? (makes domain sense)
if "log10_in_need" in coef_df["feature"].values:
    log_in_need_coef = coef_df[coef_df["feature"] == "log10_in_need"]["abs_coef"].values[0]
    sanity_checks["log10_in_need_matters"] = log_in_need_coef > 0.05
else:
    sanity_checks["log10_in_need_matters"] = False

# Check 3: No single feature dominates everything (sign of data leakage)
top_coef_share = coef_df.iloc[0]["abs_coef"] / coef_df["abs_coef"].sum()
sanity_checks["no_single_feature_dominance"] = top_coef_share < 0.8

print("\nFeature Importance Sanity Checks:")
for check, passed in sanity_checks.items():
    status = "✅" if passed else "❌"
    print(f"  {status} {check}")

# Visualization
fig, ax = plt.subplots(figsize=(10, 6))
plot_df = coef_df.head(10).sort_values("coef")
colors = ["#dc2626" if v < 0 else "#16a34a" for v in plot_df["coef"]]
ax.barh(plot_df["feature"], plot_df["coef"], color=colors, edgecolor="black", linewidth=0.5)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Feature Coefficients (Ridge)\nGreen = higher $/person | Red = lower $/person")
ax.set_xlabel("Coefficient (standardized)")
plt.tight_layout()

# Log to MLflow
with mlflow.start_run(run_name="Test_6_Feature_Importance_Sanity"):
    mlflow.set_tag("test_type", "feature_sanity")
    
    for check, passed in sanity_checks.items():
        mlflow.log_metric(check, 1.0 if passed else 0.0)
    
    # Log top features
    for i, (_, row) in enumerate(coef_df.head(5).iterrows()):
        mlflow.log_param(f"top_feature_{i+1}", row["feature"])
        mlflow.log_metric(f"top_coef_{i+1}", row["coef"])
    
    all_passed = all(sanity_checks.values())
    mlflow.log_metric("feature_sanity_pass", 1.0 if all_passed else 0.0)
    
    # Log plot
    with tempfile.TemporaryDirectory() as tmpdir:
        fig.savefig(f"{tmpdir}/feature_importance.png", dpi=150, bbox_inches="tight")
        mlflow.log_artifact(f"{tmpdir}/feature_importance.png", "plots")

plt.show()

print(f"\n{'✅ FEATURE IMPORTANCE SANITY VALIDATED' if all_passed else '⚠️ FEATURE IMPORTANCE NEEDS REVIEW'}")

In [ ]:
# ============================================================================
# CELL 8: Data Leakage Detection
# ============================================================================

print("="*70)
print("TEST 7: DATA LEAKAGE DETECTION")
print("="*70)

leakage_tests = {}

# Test 1: Train R² should not be dramatically higher than test R²
# (Large gap suggests overfitting or leakage)
train_test_gap_ridge = baseline_results["Ridge"]["train_r2"] - baseline_results["Ridge"]["test_r2"]
train_test_gap_gb = baseline_results["GradientBoosting"]["train_r2"] - baseline_results["GradientBoosting"]["test_r2"]

leakage_tests["ridge_no_severe_overfit"] = train_test_gap_ridge < 0.3
leakage_tests["gb_no_severe_overfit"] = train_test_gap_gb < 0.4  # Tree models can overfit more

print(f"Train-Test R² Gap:")
print(f"  Ridge:            {train_test_gap_ridge:.4f} {'✅' if train_test_gap_ridge < 0.3 else '❌'}")
print(f"  GradientBoosting: {train_test_gap_gb:.4f} {'✅' if train_test_gap_gb < 0.4 else '⚠️'}")

# Test 2: Test R² should be reasonable (not near 1.0 which suggests leakage)
leakage_tests["ridge_test_r2_reasonable"] = baseline_results["Ridge"]["test_r2"] < 0.95
leakage_tests["gb_test_r2_reasonable"] = baseline_results["GradientBoosting"]["test_r2"] < 0.95

print(f"\nTest R² Reasonableness (< 0.95):")
print(f"  Ridge:            {baseline_results['Ridge']['test_r2']:.4f} {'✅' if baseline_results['Ridge']['test_r2'] < 0.95 else '❌'}")
print(f"  GradientBoosting: {baseline_results['GradientBoosting']['test_r2']:.4f} {'✅' if baseline_results['GradientBoosting']['test_r2'] < 0.95 else '❌'}")

# Test 3: Check for target in features (direct leakage)
potential_leaky_features = ["usd_per_in_need", "log10_usd_per_in_need", "req_sum", "mismatch"]
leaky_in_features = [f for f in potential_leaky_features if f in num_features + cat_features]
leakage_tests["no_target_in_features"] = len(leaky_in_features) == 0

print(f"\nTarget Leakage Check:")
if leaky_in_features:
    print(f"  ❌ WARNING: Potentially leaky features found: {leaky_in_features}")
else:
    print(f"  ✅ No target-derived features in input")

# Test 4: Temporal leakage check (training on future data)
train_years_check = set(train_df["year"].unique())
test_years_check = set(test_df["year"].unique())
temporal_overlap = train_years_check.intersection(test_years_check)
leakage_tests["no_temporal_leakage"] = len(temporal_overlap) == 0

print(f"\nTemporal Leakage Check:")
print(f"  Train years: {sorted(train_years_check)}")
print(f"  Test years:  {sorted(test_years_check)}")
print(f"  Overlap:     {'✅ None' if len(temporal_overlap) == 0 else f'❌ {temporal_overlap}'}")

# Summary
print("\n" + "-"*50)
print("Leakage Test Summary:")
for test, passed in leakage_tests.items():
    status = "✅" if passed else "❌"
    print(f"  {status} {test}")

# Log to MLflow
with mlflow.start_run(run_name="Test_7_Leakage_Detection"):
    mlflow.set_tag("test_type", "leakage_detection")
    
    for test, passed in leakage_tests.items():
        mlflow.log_metric(test, 1.0 if passed else 0.0)
    
    mlflow.log_metric("train_test_gap_ridge", train_test_gap_ridge)
    mlflow.log_metric("train_test_gap_gb", train_test_gap_gb)
    
    all_passed = all(leakage_tests.values())
    mlflow.log_metric("leakage_detection_pass", 1.0 if all_passed else 0.0)

print(f"\n{'✅ NO DATA LEAKAGE DETECTED' if all_passed else '⚠️ POTENTIAL DATA LEAKAGE - INVESTIGATE'}")

In [ ]:
# ============================================================================
# CELL 9: Final Pipeline Validation Summary
# ============================================================================

print("="*80)
print("         PIPELINE VALIDATION SUMMARY")
print("="*80)

# Collect all test results
all_tests = {
    "1. Data Ingestion": all(r["status"] == "✅" for r in ingestion_results.values()),
    "2. Feature Engineering": all(v >= 0.80 for v in feature_tests.values()),
    "3. Dataset Construction": dataset_metrics["target_available"] >= 0.5,
    "4. Models Beat Baseline": (ridge_improvement > 0 or gb_improvement > 0),
    "5. CV Stability": max_std < 0.25,
    "6. Feature Sanity": all(sanity_checks.values()),
    "7. No Data Leakage": all(leakage_tests.values()),
}

print("\nTest Results:")
print("-"*50)
passed_count = 0
for test_name, passed in all_tests.items():
    status = "✅ PASS" if passed else "❌ FAIL"
    print(f"  {status}  {test_name}")
    if passed:
        passed_count += 1

print("-"*50)
print(f"\nOverall: {passed_count}/{len(all_tests)} tests passed")

# Key metrics summary
print("\n" + "="*50)
print("KEY METRICS FOR YOUR NOTEBOOK")
print("="*50)
print(f"\nGeo-Insight Model Performance:")
print(f"  Ridge Test R²:            {baseline_results['Ridge']['test_r2']:.4f}")
print(f"  GradientBoosting Test R²: {baseline_results['GradientBoosting']['test_r2']:.4f}")
print(f"  Improvement over baseline: +{max(ridge_improvement, gb_improvement):.4f}")
print(f"\nCross-Validation (5-fold):")
print(f"  Ridge Mean R²:            {cv_results['Ridge']['mean_r2']:.4f} ± {cv_results['Ridge']['std_r2']:.4f}")
print(f"  GradientBoosting Mean R²: {cv_results['GradientBoosting']['mean_r2']:.4f} ± {cv_results['GradientBoosting']['std_r2']:.4f}")

# Final verdict
overall_pass = passed_count >= 5  # Allow 2 minor failures

print("\n" + "="*80)
if overall_pass:
    print("✅ PIPELINE VALIDATION SUCCESSFUL")
    print("   Your data analysis flow is producing meaningful, non-spurious results.")
    print("   Models are learning real patterns, not just noise.")
else:
    print("⚠️ PIPELINE VALIDATION HAS WARNINGS")
    print("   Review the failed tests above before finalizing your analysis.")
print("="*80)

# Log final summary to MLflow
with mlflow.start_run(run_name="FINAL_Pipeline_Summary"):
    mlflow.set_tag("test_type", "summary")
    mlflow.set_tag("overall_status", "PASS" if overall_pass else "FAIL")
    
    for test_name, passed in all_tests.items():
        clean_name = test_name.split(".")[1].strip().replace(" ", "_").lower()
        mlflow.log_metric(f"test_{clean_name}", 1.0 if passed else 0.0)
    
    mlflow.log_metric("tests_passed", passed_count)
    mlflow.log_metric("tests_total", len(all_tests))
    mlflow.log_metric("pass_rate", passed_count / len(all_tests))
    mlflow.log_metric("overall_pass", 1.0 if overall_pass else 0.0)
    
    # Log key model metrics
    mlflow.log_metric("best_test_r2", max(baseline_results['Ridge']['test_r2'], baseline_results['GradientBoosting']['test_r2']))
    mlflow.log_metric("improvement_over_baseline", max(ridge_improvement, gb_improvement))

In [ ]:
# ============================================================================
# CELL 10: View All MLflow Runs
# ============================================================================

print("="*80)
print("         MLFLOW EXPERIMENT RUNS")
print("="*80)

# Get all runs from this experiment
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
runs_df = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

if not runs_df.empty:
    # Select key columns
    display_cols = ["run_id", "tags.test_type", "status", "start_time"]
    
    # Add any pass metrics
    pass_cols = [c for c in runs_df.columns if c.endswith("_pass") or c == "metrics.pass_rate"]
    display_cols.extend(pass_cols[:3])  # Limit to first 3
    
    available_cols = [c for c in display_cols if c in runs_df.columns]
    
    print(f"\nTotal runs: {len(runs_df)}")
    print("\nRun Summary:")
    
    # Simplified display
    for _, row in runs_df.iterrows():
        run_id = row['run_id'][:8]
        test_type = row.get('tags.test_type', 'N/A')
        status = row.get('status', 'N/A')
        print(f"  [{run_id}] {test_type:<30} Status: {status}")
    
    print("\n" + "-"*50)
    print("To view detailed results:")
    print("  Databricks: Click 'Experiments' in sidebar")
    print("  Local: Run 'mlflow ui' in terminal, open http://localhost:5000")
else:
    print("No runs found in experiment.")

---

## How to Use These Results in Your Main Notebook

### 1. Copy the Validated Metrics

Add this to your conclusions section:

```markdown
### Model Validation Results

Our pipeline was validated using MLflow experiment tracking:

- **Models beat naive baseline**: +X.XX R² improvement
- **Cross-validation stable**: Mean R² = X.XX ± X.XX (5-fold)
- **No data leakage detected**: Temporal split validated
- **Feature importance interpretable**: log10_in_need, severity_index, region
```

### 2. Reference the Experiment

In your video presentation:

> "We validated our pipeline using MLflow, confirming that our models
> learn real patterns rather than noise. Our GradientBoosting model
> achieves X.XX R² on held-out 2026 data, a significant improvement
> over naive baselines."

### 3. Screenshot the MLflow UI

Include in your video:
- Experiment comparison table
- Cross-validation box plot
- Feature importance chart

---